# Experiment 1: Design-Space Benchmark

Stacked-bar comparison of 7 MVHT configurations + SNAP + IVMH baselines.

Workloads: RH (Read-Heavy 80/20), B (Balanced 50/50), WH (Write-Heavy 20/80)
Appendix: RO (100/0), RE (95/5), WE (5/95), WO (0/100)

In [ ]:
import subprocess, re, os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# ── Config ──────────────────────────────────────────────────────────────
BIN = Path("../../../target/release/htap_wkld")
OUT_DIR = Path("output_exp1")
OUT_DIR.mkdir(exist_ok=True)

TABLE_TYPES = ["naive", "ivmh", "heap", "par", "chain"]
BASELINES = {"naive", "ivmh"}  # table types that have no repair distinction

REPEAT = 3          # ← 여기서 반복 횟수 조절
TXN_NUM = 100       # normalise duration by this

WORKLOADS = {
    # name: analytical_ratio  (update = 1 - analytical - gc)
    "RO": 1.00,  # Read-Only
    "RE": 0.95,  # Read-Extreme
    "RH": 0.80,  # Read-Heavy
    "B":  0.50,  # Balanced
    "WH": 0.20,  # Write-Heavy
    "WE": 0.05,  # Write-Extreme
    "WO": 0.00,  # Write-Only
}

# Base arguments shared across all workloads
BASE_ARGS = [
    "--update-ratio", "0.005",
    "--probe-ratio", "0.0005",
    "--txn-count", "40",
    "--scan-count", "6",
    "--warehouse-count", "5",
    "--scan-reuse-ratio", "0.8",
    "--txn-gc-ratio", "0.05",
]

## 1. Build

In [ ]:
r = subprocess.run(["cargo", "build", "--release", "--bin", "htap_wkld"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr)
    raise SystemExit(1)
print("Build OK")

## 2. Run & Collect

In [ ]:
from bench_script_functions import parse_result

TX_MAP = {"MarkTs": "BuildSnap", "DelSc": "DeltaScan"}

def run_workload(wkld_name, analytical_ratio):
    """Run all table types for a single workload, return aggregated DataFrame."""
    args = BASE_ARGS + ["--analytical-ratio", str(analytical_ratio)]
    all_dfs = []
    for tt in TABLE_TYPES:
        print(f"  {wkld_name} / {tt} ...", end=" ", flush=True)
        dfs = []
        for trial in range(REPEAT):
            cmd = [str(BIN)] + args + ["--table-type", tt]
            r = subprocess.run(cmd, capture_output=True, text=True)
            df = parse_result(r.stdout, tt)
            dfs.append(df)
        df_all = pd.concat(dfs, ignore_index=True)
        df_avg = (
            df_all
            .groupby(["idx","tx_id","tx_type","repair_type","table_type"], as_index=False)
            .agg({"duration_ms": "mean", "count": "first", "read_ts": "first",
                  "from_ts": "first", "to_ts": "first"})
        )
        all_dfs.append(df_avg)
        print("done")
    df = pd.concat(all_dfs, ignore_index=True)
    df["tx_type"] = df["tx_type"].replace(TX_MAP)

    # Aggregate to (table_type, repair_type, tx_type) -> total duration
    result = df.groupby(["table_type","repair_type","tx_type"], as_index=False)["duration_ms"].sum()
    result["duration_ms"] = result["duration_ms"] / TXN_NUM

    # For baselines (naive, ivmh): all repair modes produce identical results,
    # so keep only "Write Repair" rows and relabel to "" to avoid 3x duplication.
    baseline_mask = result["table_type"].isin(BASELINES)
    keep = ~baseline_mask | (result["repair_type"] == "Write Repair")
    result = result[keep].copy()
    result.loc[result["table_type"].isin(BASELINES), "repair_type"] = ""
    return result

# ── Run only missing workloads (skip if CSV already exists) ────────────
results = {}
any_run = False
for name, ar in WORKLOADS.items():
    csv_path = OUT_DIR / f"exp1_{name}.csv"
    if csv_path.exists():
        print(f"  {name}: CSV exists, skipping run")
        continue
    any_run = True
    print(f"=== Workload: {name} (analytical={ar}) ===")
    df = run_workload(name, ar)
    df.to_csv(csv_path, index=False)
    results[name] = df
    print(f"  Saved to {csv_path}")

if not any_run:
    print("All CSVs already exist — no experiments to run.")
else:
    print("\nDone.")

## 3. Plot

In [ ]:
# ── Plotting function ──────────────────────────────────────────────────

# Paul Tol Bright
TOL = {
    "blue": "#4477AA", "cyan": "#66CCEE", "green": "#228833",
    "yellow": "#CCBB44", "red": "#EE6677", "purple": "#AA3377",
    "grey": "#BBBBBB", "darkgreen": "#8C9916",
}
COLOR_MAP = {
    "InitLoad": TOL["grey"], "Update": TOL["yellow"],
    "GbgCollect": TOL["cyan"], "BuildSnap": TOL["purple"],
    "Probe": TOL["red"], "RecentScan": TOL["green"],
    "HistoryScan": TOL["darkgreen"], "DeltaScan": TOL["blue"],
}
HATCH_MAP = {
    "DeltaScan": "\\\\\\\\",
    "Probe": "///",
    "HistoryScan": "\\\\\\\\",
    "RecentScan": "///",
}
TX_ORDER = ["InitLoad","Update","GbgCollect","BuildSnap",
            "RecentScan","HistoryScan","Probe","DeltaScan"]
WRITE_OPS = {"InitLoad","Update","GbgCollect","BuildSnap"}

# Bar layout: SNAP | IVMH | MONO(NR/RR/WR) | DUAL(WR) | EPOCH(NR/RR/WR)
TABLE_ORDER = ["naive", "ivmh", "heap", "chain", "par"]
TABLE_DISPLAY = {"naive":"SNAP", "ivmh":"IVMH", "heap":"MONO", "chain":"DUAL", "par":"EPOCH"}
REPAIR_ORDER = {
    "naive": [""],
    "ivmh":  [""],
    "heap":  ["No Repair","Read Repair","Write Repair"],
    "chain": ["Write Repair"],
    "par":   ["No Repair","Read Repair","Write Repair"],
}
REPAIR_DISPLAY = {"No Repair":"NR","Read Repair":"RR","Write Repair":"WR","":""}

plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman","Times","DejaVu Serif"]
plt.rcParams["hatch.linewidth"] = 0.9


def plot_exp1(df, title, ylim_top=None):
    pvt = (
        df.pivot_table(index=["table_type","repair_type"],
                       columns="tx_type", values="duration_ms",
                       aggfunc="sum", fill_value=0.0)
        .reindex(columns=TX_ORDER, fill_value=0.0)
    )

    bar_w = 0.58
    gap = 0.36
    bar_x, minor_labels, major_centers, major_labels, combos = [], [], [], [], []
    x = 0.0
    for t in TABLE_ORDER:
        subs = REPAIR_ORDER[t]
        start = x
        for r in subs:
            bar_x.append(x)
            minor_labels.append(REPAIR_DISPLAY.get(r, ""))
            combos.append((t, r))
            x += 0.78
        major_centers.append((start + x - 1.0) / 2.0)
        major_labels.append(TABLE_DISPLAY[t])
        x += gap

    fig, ax = plt.subplots(figsize=(7.8, 5.5))
    fig.subplots_adjust(bottom=0.22)

    for i, (t, r) in enumerate(combos):
        row = pvt.loc[(t, r)] if (t, r) in pvt.index else pd.Series(0.0, index=TX_ORDER)
        bottom = 0.0
        for tx in TX_ORDER:
            v = float(row.get(tx, 0.0))
            if v <= 0:
                continue
            if tx in WRITE_OPS:
                ax.bar(bar_x[i], v, bottom=bottom, width=bar_w,
                       color=COLOR_MAP[tx], edgecolor="black", linewidth=0.4, zorder=2)
            else:
                ax.bar(bar_x[i], v, bottom=bottom, width=bar_w,
                       color="white", edgecolor="black", linewidth=0.4, zorder=2)
                ax.bar(bar_x[i], v, bottom=bottom, width=bar_w,
                       color="none", edgecolor=COLOR_MAP[tx], linewidth=0.9,
                       hatch=HATCH_MAP.get(tx, "///"), zorder=3)
            bottom += v

    ax.set_xticks([])
    for xi, lab in zip(bar_x, minor_labels):
        ax.text(xi, -0.028, lab, ha="center", va="top",
                transform=ax.get_xaxis_transform(), fontsize=10, clip_on=False)
    for xc, lab in zip(major_centers, major_labels):
        ax.text(xc, -0.072, lab, ha="center", va="top",
                transform=ax.get_xaxis_transform(), fontsize=12, clip_on=False)

    for k in range(1, len(major_centers)):
        prev_end = max(xx for xx,(tt,_) in zip(bar_x, combos) if tt == TABLE_ORDER[k-1])
        next_start = min(xx for xx,(tt,_) in zip(bar_x, combos) if tt == TABLE_ORDER[k])
        ax.axvline((prev_end + next_start)/2, linestyle=":", linewidth=0.6, alpha=0.3)

    ax.set_axisbelow(True)
    ax.yaxis.grid(True, linestyle="--", linewidth=0.6, alpha=0.6)
    ax.set_ylabel("Duration (ms)")
    if ylim_top:
        ax.set_ylim(top=ylim_top)

    handles, labels = [], []
    for tx in TX_ORDER[::-1]:
        if tx in WRITE_OPS:
            p = Patch(facecolor=COLOR_MAP[tx], edgecolor="black", linewidth=0.4)
        else:
            p = Patch(facecolor="white", edgecolor=COLOR_MAP[tx],
                      hatch=HATCH_MAP.get(tx, "///"), linewidth=0.9)
        handles.append(p)
        labels.append(tx)
    ax.legend(handles, labels, title="Operations",
              loc="upper right", framealpha=0.9, fontsize=8.5, title_fontsize=8.5)

    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{title}.pdf", format="pdf")
    plt.show()

In [ ]:
# ── Generate plots ─────────────────────────────────────────────────────
# Main text: RH, B, WH
# Appendix: RO, RE, WE, WO

for name in ["RH", "B", "WH", "RO", "RE", "WE", "WO"]:
    csv_path = OUT_DIR / f"exp1_{name}.csv"
    if csv_path.exists():
        # keep_default_na=False: prevent pandas from turning "" into NaN
        df = pd.read_csv(csv_path, keep_default_na=False)
    else:
        df = results.get(name)
    if df is not None:
        df["repair_type"] = df["repair_type"].fillna("")  # safety net
        print(f"\n=== {name} ===")
        plot_exp1(df, title=name)